In [ ]:
import os,sys,json,hashlib,zipfile,io,subprocess,shutil,time
from pathlib import Path
from datetime import datetime,timezone
RUN_DIR=Path('/kaggle/working/s1_e1_01')/'cpu'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True,exist_ok=False)
SOURCE=RUN_DIR/'source'
payload=__import__('base64').b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='1d3582462b2b84021c36172fc86215bd40387dba1d39d2fa4cd2b1e3334a9e52'
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert z.testzip() is None
    assert all(not n.startswith('/') and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(SOURCE)
sys.path.insert(0,str(SOURCE))
REG=json.loads((SOURCE/'s1/registration.json').read_text())
for n,digest in REG['source_hashes'].items():assert hashlib.sha256((SOURCE/n).read_bytes()).hexdigest()==digest,n
WHEELHOUSE_DIR=next(Path('/kaggle/input').rglob('swegemma-0.2.11-py3-none-any.whl')).parent
actual={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in WHEELHOUSE_DIR.glob('*.whl') if 'cutlass' not in p.name.lower()}
assert actual==REG['wheel_hashes'],'Wheel bytes drift'
(RUN_DIR/'wheel_hashes.json').write_text(json.dumps(actual,indent=2))
os.environ.update(VLLM_WORKER_MULTIPROC_METHOD='spawn',VLLM_NO_USAGE_STATS='1',VLLM_ENGINE_READY_TIMEOUT_S='1200',VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS='1',NCCL_DEBUG='WARN',PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',LOCAL_API_KEY='EMPTY',OPENAI_API_KEY='EMPTY')


In [ ]:
import glob
os.environ.update(TRANSFORMERS_NO_TF='1',TRANSFORMERS_NO_FLAX='1',TRANSFORMERS_OFFLINE='1',HF_HUB_OFFLINE='1',LITELLM_LOCAL_MODEL_COST_MAP='True',LITELLM_TELEMETRY='False',OTEL_SDK_DISABLED='true',PYTHONIOENCODING='utf-8')
for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth','/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for path in glob.glob(pattern):Path(path).unlink(missing_ok=True)
links=RUN_DIR/'wheelhouse_links';links.mkdir()
try:
    for p in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in p.name.lower():continue
        name=p.name.replace('cu128','+cu128') if 'cu128' in p.name and '+' not in p.name else p.name
        (links/name).symlink_to(p)
    with (RUN_DIR/'installation.log').open('x') as log:
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall']+sorted(str(p) for p in links.glob('*.whl')),stdout=log,stderr=subprocess.STDOUT,check=True,timeout=1800)
finally:shutil.rmtree(links)


In [ ]:
from harness_v2.evidence import bundle
try:
    p=subprocess.run([sys.executable,'-m','s1.cpu',str(RUN_DIR/'validation')],cwd=SOURCE,capture_output=True,text=True,timeout=1500)
    (RUN_DIR/'runner.stdout.log').write_text(p.stdout);(RUN_DIR/'runner.stderr.log').write_text(p.stderr)
    print(p.stdout+p.stderr,flush=True)
    assert p.returncode==0,'CPU gate failed; GPU forbidden'
    from s1.task_identity import identify,find_data
    selection=[i for items in REG['selected'].values() for i in items]
    (RUN_DIR/'task_inputs.json').write_text(json.dumps(identify(find_data(),selection),indent=2))
finally:print('S1_CPU_EVIDENCE',bundle(RUN_DIR),flush=True)
